<a href="https://colab.research.google.com/github/SunnyZhao2004/Replication/blob/master/0810match_pdf_to_excel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import pandas as pd
from pathlib import Path


BASE_DIR   = "/content/drive/MyDrive/Find AR in The Excel File"
CSV_INPUT  = f"{BASE_DIR}/Large Excel File/combined_excel_files.csv"
PDF_DIR = Path(f"{BASE_DIR}/PDF Files/Selected 400 ARs (for date verification)")
OUT_CSV    = Path(BASE_DIR) / "matched.csv"

pdf_paths = sorted(PDF_DIR.rglob("*.pdf"))

for p in pdf_paths:
    fname = p.name
    base = fname[:-4]           # drop .pdf
    parts = base.split('-')
    if len(parts) != 7:
        print(fname)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2000-01-12-Pliva Farm-Societe Generale-54448886.pdf
2000-04-16-TransAlta -Raymond James Ltd. (-65289401.pdf
2000-04-18-EarthLink -Jefferies-4148958.pdf
2000-10-02-ITT Corp-Deutsche Bank-4800474.pdf
2001-02-22-Medtronic -Piper Jaffray-54166949.pdf
2001-10-04-Hanwha Oce-JPMorgan-52314503.pdf
2005-02-11-PD.N^C07-Freeport-M-RBC Capital Markets-34404674.pdf
2006-05-17-Swiss Life-Warburg Research Gmb-59867671.pdf
2009-02-24-UBS Equities-44695623.pdf
2009-04-22-AFLT.MM-Aeroflot-R-Auerbach Grayson  C-45175213.pdf
2009-11-02-Agnico Eag-RBC Capital Markets-46832821.pdf
2010-06-17-Cisco Syst-EVERCORE ISI-48717141.pdf
2010-12-22-Societe Generale-51310323.pdf
2010-12-29-Whitecap R-Macquarie Research-51345278.pdf
2011-08-09-CCL.AX^E21-Coca-Cola -Nomura-56914621.pdf
2011-11-18-Deutsche W-UBS Equities-57867375.pdf
2012-03-01-GFT Techno-Warburg Research Gmb-58936142.pdf
2013-

In [ ]:
rows = []
pdf = 0
for name in os.listdir(PDF_DIR):
    pdf+=1
    if not name.lower().endswith(".pdf"):
        continue  # skip non-pdf files

    base = name[:-4]              # drop ".pdf"
    parts = base.split('-')

    # keep only filenames that split into exactly 7 parts
    if len(parts) != 7:
        continue

    # extract fields
    date = "-".join(parts[0:3])   # YYYY-MM-DD
    ticker = parts[3]
    remainder = parts[4:]
    dcn = remainder[-1]
    contributor = remainder[-2]
    company = "-".join(remainder[:-2])  # may contain hyphens

    rows.append({
        "date": date,
        "ticker": ticker,
        "company": company,
        "contributor": contributor,
        "DCN": dcn,
        "pdf_filename": name
    })

# df is the dataframe for pdf file names
df = pd.DataFrame(rows, columns=["date", "ticker", "company", "contributor", "DCN", "pdf_filename"])

print("Rows kept:", len(df))
print(df.head())

out_csv = os.path.join(PDF_DIR, "parsed_pdfs.csv")
df.to_csv(out_csv, index=False)
print("Saved to:", out_csv)

Rows kept: 371
         date   ticker     company     contributor        DCN  \
0  2019-08-02     BR.N  Broadridge  Wolfe Research   85883078   
1  2015-04-29   SAVE.N  Spirit Air  Wolfe Research   70087532   
2  2024-05-17    DXC.N  DXC Techno  Wolfe Research  108267033   
3  2023-07-20  FITB.OQ  Fifth Thir  Wolfe Research  102974421   
4  2021-04-22    HCA.N  HCA Health  Wolfe Research   92066263   

                                        pdf_filename  
0  2019-08-02-BR.N-Broadridge-Wolfe Research-8588...  
1  2015-04-29-SAVE.N-Spirit Air-Wolfe Research-70...  
2  2024-05-17-DXC.N-DXC Techno-Wolfe Research-108...  
3  2023-07-20-FITB.OQ-Fifth Thir-Wolfe Research-1...  
4  2021-04-22-HCA.N-HCA Health-Wolfe Research-920...  
Saved to: /content/drive/MyDrive/Find AR in The Excel File/PDF Files/Selected 400 ARs (for date verification)/parsed_pdfs.csv


In [ ]:
df['DCN'] = df['DCN'].astype(str).str.replace(r'\.undefined(\s*\(.*\))?$', '', regex=True)
df['DCN'] = df['DCN'].astype(int)

In [ ]:
excel = pd.read_csv("/content/drive/MyDrive/Find AR in The Excel File/Large Excel File/combined_excel_files.csv")

In [ ]:
merged = excel.merge(df,on = "DCN",how = 'outer',indicator=True)

In [ ]:
excel['DCN'] = excel['DCN'].astype(str)
df['DCN'] = df['DCN'].astype(str)

mask = excel['DCN'].apply(lambda x: any(dcn in x for dcn in df['DCN']))
matched_excel = excel[mask]

In [ ]:
# Make sure both are strings
excel['DCN'] = excel['DCN'].astype(str)
df['DCN']    = df['DCN'].astype(str)

# Small list to scan (df is small)
small_dcns = df['DCN'].dropna().astype(str).unique().tolist()

# Mask: keep excel rows where its DCN is a substring of ANY df DCN
mask = excel['DCN'].apply(lambda x: any(x in d for d in small_dcns))
matched_excel2 = excel[mask].copy()


In [ ]:
merged['_merge'].value_counts()

,count
_merge,
left_only,1073114
right_only,349
both,22


In [ ]:
merged[merged['_merge'] =='both'].to_csv('/content/drive/MyDrive/Find AR in The Excel File/merged.csv')

In [ ]:
# manually update rows with part != 7, uplaod the updated file